# GitHub Repository Auditor

This AI Agent audits Github repos in real time. Given a repository's owner and name, the agent will inspect structure, analyze dependencies, detect known CVEs and make a report with a list of priorized actions to do.

## Import Libraries

In [1]:
import requests
import json
import os
import base64
from typing import List, Optional
from dotenv import load_dotenv
import openai

In [2]:
load_dotenv()

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
GITHUB_API = "https://api.github.com"
OSV_API = "https://api.osv.dev/v1"

GITHUB_HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28"
}

## Tool Functions

1. `get_repo_info`: repository's general metadata
2. `get_file_tree`: file directory complete structure
3. `get_file_content`: specific file contents
4. `get_dependencies`: project dependencies by language
5. `check_vulnerabilities`: known CVEs via OSV.dev

In [3]:
def get_repo_info(owner: str, repo: str) -> str:
    """
    Fetch general metadata about a GitHub repository.

    Args:
        owner: Repository owner (user or organization)
        repo: Repository name

    Returns:
        JSON string with repo metadata
    """
    url = f"{GITHUB_API}/repos/{owner}/{repo}"
    response = requests.get(url, headers=GITHUB_HEADERS)
    response.raise_for_status()
    data = response.json()

    return json.dumps({
        "name": data["full_name"],
        "description": data.get("description"),
        "language": data.get("language"),
        "languages_url": data.get("languages_url"),
        "license": data["license"]["name"] if data.get("license") else None,
        "stars": data["stargazers_count"],
        "forks": data["forks_count"],
        "open_issues": data["open_issues_count"],
        "default_branch": data["default_branch"],
        "created_at": data["created_at"][:10],
        "last_push": data["pushed_at"][:10],
        "size_kb": data["size"],
        "has_wiki": data["has_wiki"],
        "has_issues": data["has_issues"],
        "topics": data.get("topics", [])
    }, indent=2)

In [4]:
get_repo_info("psf", "requests")

'{\n  "name": "psf/requests",\n  "description": "A simple, yet elegant, HTTP library.",\n  "language": "Python",\n  "languages_url": "https://api.github.com/repos/psf/requests/languages",\n  "license": "Apache License 2.0",\n  "stars": 53900,\n  "forks": 9795,\n  "open_issues": 301,\n  "default_branch": "main",\n  "created_at": "2011-02-13",\n  "last_push": "2026-03-19",\n  "size_kb": 13312,\n  "has_wiki": true,\n  "has_issues": true,\n  "topics": [\n    "client",\n    "cookies",\n    "forhumans",\n    "http",\n    "humans",\n    "python",\n    "python-requests",\n    "requests"\n  ]\n}'

In [5]:
def get_file_tree(owner: str, repo: str, max_files: int = 100) -> str:
    """
    Fetch the full file/directory tree of a repository.

    Args:
        owner: Repository owner
        repo: Repository name
        max_files: Maximum number of file paths to return (default: 100)

    Returns:
        JSON string with file tree and structural analysis
    """
    url = f"{GITHUB_API}/repos/{owner}/{repo}/git/trees/HEAD"
    response = requests.get(url, headers=GITHUB_HEADERS, params={"recursive": "1"})
    response.raise_for_status()
    data = response.json()

    all_paths = [item["path"] for item in data.get("tree", [])]
    truncated = data.get("truncated", False)

    # Structural signals useful for the agent to reason about
    has_tests = any(
        p.startswith(("test", "tests", "__tests__", "spec", "specs"))
        or "/test" in p or "_test." in p or ".test." in p or ".spec." in p
        for p in all_paths
    )
    has_ci = any(
        p.startswith(".github/workflows") or p in (".travis.yml", "Jenkinsfile", ".circleci/config.yml")
        for p in all_paths
    )
    has_docker = any(p in ("Dockerfile", "docker-compose.yml", "docker-compose.yaml") for p in all_paths)
    has_readme = any(p.lower().startswith("readme") for p in all_paths)
    has_contributing = any("contributing" in p.lower() for p in all_paths)
    has_env_example = any(".env.example" in p or ".env.sample" in p for p in all_paths)
    has_gitignore = ".gitignore" in all_paths

    dependency_files = [
        p for p in all_paths
        if p in (
            "requirements.txt", "requirements-dev.txt", "Pipfile", "pyproject.toml",
            "package.json", "package-lock.json", "yarn.lock",
            "pom.xml", "build.gradle", "Gemfile"
        )
    ]

    return json.dumps({
        "total_files": len(all_paths),
        "truncated": truncated,
        "paths": all_paths[:max_files],
        "signals": {
            "has_tests": has_tests,
            "has_ci": has_ci,
            "has_docker": has_docker,
            "has_readme": has_readme,
            "has_contributing_guide": has_contributing,
            "has_env_example": has_env_example,
            "has_gitignore": has_gitignore,
            "dependency_files_found": dependency_files
        }
    }, indent=2)

In [6]:
get_file_tree("psf", "requests")

'{\n  "total_files": 155,\n  "truncated": false,\n  "paths": [\n    ".coveragerc",\n    ".git-blame-ignore-revs",\n    ".github",\n    ".github/CODEOWNERS",\n    ".github/CODE_OF_CONDUCT.md",\n    ".github/CONTRIBUTING.md",\n    ".github/FUNDING.yml",\n    ".github/ISSUE_TEMPLATE.md",\n    ".github/ISSUE_TEMPLATE",\n    ".github/ISSUE_TEMPLATE/Bug_report.md",\n    ".github/ISSUE_TEMPLATE/Custom.md",\n    ".github/ISSUE_TEMPLATE/Feature_request.md",\n    ".github/SECURITY.md",\n    ".github/dependabot.yml",\n    ".github/workflows",\n    ".github/workflows/close-issues.yml",\n    ".github/workflows/codeql-analysis.yml",\n    ".github/workflows/lint.yml",\n    ".github/workflows/lock-issues.yml",\n    ".github/workflows/publish.yml",\n    ".github/workflows/run-tests.yml",\n    ".gitignore",\n    ".pre-commit-config.yaml",\n    ".readthedocs.yaml",\n    "AUTHORS.rst",\n    "HISTORY.md",\n    "LICENSE",\n    "MANIFEST.in",\n    "Makefile",\n    "NOTICE",\n    "README.md",\n    "docs",\n  

## Tool Schema

Here are the schema of each tool which you will provide to the LLM.

In [13]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_repo_info",
            "description": "Fetch general metadata about a GitHub repository: language, license, stars, last activity, open issues, topics.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner (user or organization)"},
                    "repo": {"type": "string", "description": "Repository name"}
                },
                "required": ["owner", "repo"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_file_tree",
            "description": "Fetch the full file and directory structure of a repository. Returns structural signals like presence of tests, CI config, Docker, README, and dependency files.",
            "parameters": {
                "type": "object",
                "properties": {
                    "owner": {"type": "string", "description": "Repository owner"},
                    "repo": {"type": "string", "description": "Repository name"},
                    "max_files": {"type": "integer", "description": "Max number of paths to return (default: 100)", "default": 100}
                },
                "required": ["owner", "repo"]
            }
        }
    },
]

## Tool Mapping

This code handles tool mapping and execution.

In [14]:
mapping_tool_function = {
    "get_repo_info": get_repo_info,
    "get_file_tree": get_file_tree
}

def execute_tool(tool_name, tool_args):
    if isinstance(tool_args, str):
        tool_args = json.loads(tool_args)

    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."

    elif isinstance(result, list):
        result = ', '.join(result)

    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)

    else:
        # For any other type, convert using str()
        result = str(result)
    return result

## Chatbot Code

The chatbot handles the user's queries one by one, but it does not persist memory across the queries.

In [32]:
client = openai.OpenAI()
history = []

history = [
    {
        "role": "system",
        "content": (
            "You are a GitHub repository auditor. "
            "When the user mentions a repository, always identify the owner and repo name. "
            "Repositories are usually written as 'owner/repo' (e.g. 'vercel/next.js'). "
            "If the user provides them separately or informally, infer both before calling any tool. "
            "If you cannot determine the owner, ask the user before proceeding."
        )
    }
]

### Query Processing

In [33]:
def process_query(query):
    global history
    
    history.append({'role': 'user', 'content': query})
    
    response = client.chat.completions.create(
        model = 'gpt-4o-mini',
        messages = history,
        tools = tools,
        max_completion_tokens = 500
    )

    process_query_flag = True
    while process_query_flag:

        message = response.choices[0].message

        if not message.tool_calls:
            print(message.content)
            history.append({'role': 'assistant', 'content': message.content})
            process_query_flag = False
            continue

        history.append({
            'role': 'assistant',
            'content': message.content or "",
            "tool_calls": message.tool_calls
        })

        for tool_call in message.tool_calls:

            tool_id = tool_call.id
            tool_args = json.loads(tool_call.function.arguments)
            tool_name = tool_call.function.name

            print(f"Calling tool {tool_name}({tool_args})")
            result = execute_tool(tool_name, tool_args)

            history.append({
                "role": "tool",
                "tool_call_id": tool_id,
                "content": json.dumps(result),
            })

        response = client.chat.completions.create(
            model = 'gpt-4o-mini',
            messages = history,
            tools = tools,
            max_completion_tokens = 500
        )

### Chat Loop

In [34]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break

            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

Feel free to interact with the chatbot. Here's an example query:

- Search for 2 papers on "LLM optimization"

In [35]:
chat_loop()

Type your queries or 'quit' to exit.
¡Hola Simón! Soy un auditor de repositorios de GitHub y puedo ayudarte a obtener información sobre repositorios, como su metadata (lenguaje, licencia, estrellas, etc.) y la estructura de archivos y directorios. Si tienes un repositorio en mente, simplemente menciona su nombre y te proporcionaré la información que necesites.


Sí, tu nombre es Simón. ¿Hay algo específico en lo que te gustaría que te ayude con respecto a un repositorio de GitHub?


